# RadioML 2018.01A Dataset Preview

This notebook previews the lazy-loading RadioML dataset wrapper used by the project. It is safe to open even before the real dataset is present; cells that need the HDF5 file will simply stop with a clear message.


## What This Notebook Covers

- verify the expected HDF5 path
- build a lazy `RadioML2018Dataset`
- inspect class names and SNR values
- plot one IQ waveform
- plot amplitude, phase, and an STFT spectrogram


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from rfml.data.radioml2018 import RadioML2018Dataset, build_label_name_map, infer_class_names_from_h5
from rfml.data.transforms import STFTTransform

H5_PATH = REPO_ROOT / 'data' / 'GOLD_XYZ_OSC.0001_1024.hdf5'
H5_PATH


In [ ]:
if not H5_PATH.exists():
    raise FileNotFoundError(
        f'Dataset not found: {H5_PATH}\n'
        'Place GOLD_XYZ_OSC.0001_1024.hdf5 under data/ before running preview cells.'
    )

class_names = infer_class_names_from_h5(H5_PATH)
dataset = RadioML2018Dataset(H5_PATH, class_names=class_names, max_samples=4096)
label_map = build_label_name_map(dataset.num_classes, class_names)
summary = dataset.describe()
summary


In [ ]:
print('num_selected_samples:', summary['num_selected_samples'])
print('num_total_samples:', summary['num_total_samples'])
print('num_classes:', summary['num_classes'])
print('snr_values:', summary['snr_values'])
print('first_five_class_counts:', list(summary['class_counts'].items())[:5])


In [ ]:
sample = dataset[0]
iq = sample['iq'].numpy()
label = int(sample['label'].item())
snr = float(sample['snr'].item())
complex_signal = iq[0] + 1j * iq[1]
amplitude = np.abs(complex_signal)
phase = np.angle(complex_signal)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(iq[0], label='I', linewidth=0.8)
axes[0].plot(iq[1], label='Q', linewidth=0.8)
axes[0].set_title(f'IQ Waveform | {label_map[label]} | SNR={snr:g} dB')
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(amplitude, color='tab:orange', linewidth=0.8)
axes[1].set_title('Amplitude')
axes[1].grid(alpha=0.25)

axes[2].plot(phase, color='tab:green', linewidth=0.8)
axes[2].set_title('Phase')
axes[2].grid(alpha=0.25)
axes[2].set_xlabel('time index')
fig.tight_layout()


In [ ]:
stft = STFTTransform(n_fft=128, hop_length=32, output='log_power', backend='torch')
spec = stft(sample['iq']).numpy()[0]

plt.figure(figsize=(10, 4))
plt.imshow(spec, aspect='auto', origin='lower', cmap='magma')
plt.title(f'STFT Spectrogram | {label_map[label]} | SNR={snr:g} dB')
plt.xlabel('time frame')
plt.ylabel('frequency bin')
plt.colorbar(label='log power')
plt.tight_layout()


## Next Steps

- run `python scripts/inspect_dataset.py --h5 data/GOLD_XYZ_OSC.0001_1024.hdf5 --max-samples 4096`
- run `python scripts/make_splits.py --h5 data/GOLD_XYZ_OSC.0001_1024.hdf5 --out outputs/splits/radioml2018_seed42.npz --seed 42`
- train `cnn1d`, `resnet1d`, and `stft_cnn` with the same split for fair comparison
